<a href="https://colab.research.google.com/github/surue-s/BreakFixPro/blob/main/4k_Video_Upscaler_Colab_(Real_ESRGAN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4k Video Upscaler Colab (Real-ESRGAN)

Adapted from: [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN)

Made with ❤️ by: [yuvraj108c](https://github.com/yuvraj108c)

Github repository: https://github.com/yuvraj108c/4k-video-upscaler-colab

# 1. Setup (~1 minute)

In [9]:
import os, cv2, subprocess, torch

assert torch.cuda.is_available(), "GPU not detected"

# Clean install
!rm -rf Real-ESRGAN
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN

# Install deps (NO torch / torchvision downgrade)
!pip install -q basicsr facexlib gfpgan ffmpeg-python
!pip install -q -r requirements.txt

# Install package
!pip install -e .

Cloning into 'Real-ESRGAN'...
remote: Enumerating objects: 759, done.
remote: Total 759 (delta 0), reused 0 (delta 0), pack-reused 759 (from 1)
Receiving objects: 100% (759/759), 5.39 MiB | 7.86 MiB/s, done.
Resolving deltas: 100% (408/408), done.
/content/Real-ESRGAN/Real-ESRGAN
Obtaining file:///content/Real-ESRGAN/Real-ESRGAN
  Preparing metadata (setup.py) ... done
  Attempting uninstall: realesrgan
    Found existing installation: realesrgan 0.3.0
    Uninstalling realesrgan-0.3.0:
      Successfully uninstalled realesrgan-0.3.0
  Running setup.py develop for realesrgan


In [11]:
import fileinput

file_path = "/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py"

for line in fileinput.input(file_path, inplace=True):
    if "functional_tensor" in line:
        print("from torchvision.transforms.functional import rgb_to_grayscale")
    else:
        print(line, end="")


# 2. Mount drive (optional)

In [2]:
from google.colab import drive
mount_drive=False #@param{type:"boolean"}

if mount_drive:
  drive.mount('/content/gdrive/')

# 3. Upscale video

- The upscaled video will be saved to `output_dir`
- If google drive is mounted, it will be also saved at `MyDrive/Upscaled Videos (REAL-ESRGAN)`


In [12]:
import os, cv2, subprocess

video_path = "/content/nr6853bax9rmy0cx416rs4meec_result_.mp4"
output_dir = "/content/Thisdaoutpt/"
resolution = "2k (2560 x 1440)"
model = "RealESRGAN_x4plus"

os.makedirs(output_dir, exist_ok=True)
assert os.path.exists(video_path), "Video file does not exist"

video_capture = cv2.VideoCapture(video_path)
video_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

aspect_ratio = video_width / video_height

# 🎯 Resolution selection
match resolution:
    case "FHD (1920 x 1080)":
        final_width, final_height = 1920, 1080
    case "2k (2560 x 1440)":
        final_width, final_height = 2560, 1440
    case "4k (3840 x 2160)":
        final_width, final_height = 3840, 2160
    case "2 x original":
        final_width, final_height = 2*video_width, 2*video_height
    case "3 x original":
        final_width, final_height = 3*video_width, 3*video_height
    case "4 x original":
        final_width, final_height = 4*video_width, 4*video_height

# Fix orientation
if aspect_ratio < 1.0 and "original" not in resolution:
    final_width, final_height = final_height, final_width

# Scale factor
scale_factor = max(final_width/video_width, final_height/video_height)

# Ensure even dimensions
while int(video_width * scale_factor) % 2 != 0 or int(video_height * scale_factor) % 2 != 0:
    scale_factor += 0.01

print(f"Upscaling from {video_width}x{video_height} → {final_width}x{final_height}, scale={scale_factor:.2f}")

# 🚀 Run Real-ESRGAN
subprocess.run(f"""
python inference_realesrgan_video.py \
-n {model} \
-i "{video_path}" \
-o "{output_dir}" \
--outscale {scale_factor}
""", shell=True)

video_name = os.path.basename(video_path).replace(".mp4","")

# ✅ FIXED PATHS
upscaled_video_path = os.path.join(output_dir, f"{video_name}_out.mp4")
final_video_path = os.path.join(output_dir, f"{video_name}_upscaled_{final_width}_{final_height}.mp4")

# ✅ Ensure upscaling worked
assert os.path.exists(upscaled_video_path), "Upscaling failed!"

# 🎬 Crop (NO CUDA → more compatible)
if "original" not in resolution:
    print("Cropping to fit...")
    subprocess.run(f"""
    ffmpeg -y -i "{upscaled_video_path}" \
    -vf "crop={final_width}:{final_height}:(in_w-{final_width})/2:(in_h-{final_height})/2" \
    -c:v libx264 -pix_fmt yuv420p "{final_video_path}"
    """, shell=True)
else:
    subprocess.run(f'cp "{upscaled_video_path}" "{final_video_path}"', shell=True)

print(f"✅ Final video: {final_video_path}")

# Cleanup
if os.path.exists(upscaled_video_path):
    os.remove(upscaled_video_path)

Upscaling from 1280x720 → 2560x1440, scale=2.00
Cropping to fit...
✅ Final video: /content/Thisdaoutpt/nr6853bax9rmy0cx416rs4meec_result__upscaled_2560_1440.mp4


# 4. Disconnect runtime

In [ ]:
from google.colab import runtime

disconnect_when_finish = False  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()